1. Vecinos con window=5 (palabra servicio, min_count=1, seed=42)
en (0.307), las (0.298), frustrante (0.209), claro (0.190), el (0.170), confuso (0.161), terminó (0.144), registró (0.121), funcionó (0.109), expediente (0.108). Vocabulario: 54. Cobertura: 1.0.

2. Parámetro cambiado
Solo window, de 5 a 2, en la línea 26 de scripts/embeddings_network.py. Los demás parámetros (vector_size=60, min_count=1, workers=1, seed=42, epochs=30) no se tocaron.

3. Comparación tras ejecutar de nuevo (window=2)
Los 10 vecinos son los mismos y en el mismo orden. Solo cambian los decimales (por ejemplo, en: 0.307 → 0.306). El vocabulario (54) y la cobertura (1.0) no cambian.

4. Por qué el cambio modifica el tipo de contexto
window define cuántas palabras a cada lado cuentan como contexto. Una ventana pequeña captura el uso inmediato de la palabra (contexto sintáctico). Una ventana grande captura el tema de la frase (contexto asociativo). Aquí no se observó diferencia porque las frases del corpus son muy cortas, y ambas ventanas abarcan casi el mismo contexto. La hipótesis (ventana menor, vecinos más funcionales) no se confirmó con estos datos.

5. Valor seleccionado
Se restaura window=5, el valor original, porque window=2 no mejoró los resultados ni cambió la cobertura. Restaurarlo:

Copy-Item scripts\embeddings_network_original.py scripts\embeddings_network.py -Force
Si prefieres dejar window=2, escribe en su lugar: “Se mantiene window=2, sin diferencias medibles”.

El paso permite cambiar window o min_count. Si tu profesor espera un efecto visible, prueba también min_count=2: cambiará el vocabulario y la cobertura, y mostrará un contraste real. Si lo haces, pégame la salida y añado ese segundo experimento a la tabla.



In [5]:
#Paso 6. Comparar centralidades

import networkx as nx
import pandas as pd

G = nx.karate_club_graph()
tabla = pd.DataFrame({
    "grado": nx.degree_centrality(G),
    "betweenness": nx.betweenness_centrality(G),
    "pagerank": nx.pagerank(G),
}).round(3)
print(tabla.sort_values("pagerank", ascending=False).head(10))

    grado  betweenness  pagerank
33  0.515        0.304     0.097
0   0.485        0.438     0.089
32  0.364        0.145     0.076
2   0.303        0.144     0.063
1   0.273        0.054     0.057
31  0.182        0.138     0.042
23  0.152        0.018     0.041
3   0.182        0.012     0.037
5   0.121        0.030     0.034
8   0.152        0.056     0.033


In [ ]:
Nodo	Grado	Betweenness	PageRank	Lectura prudente
33	0.515	0.304	0.097	Mayor número de conexiones directas y mayor PageRank. También tiene betweenness alta.
0	0.485	0.438	0.089	La mayor betweenness: muchos caminos más cortos pasan por él.
32	0.364	0.145	0.076	Grado moderado y betweenness moderada.
2	0.303	0.144	0.063	Similar al 32, algo menos conectado.
1	0.273	0.054	0.057	Grado moderado y betweenness baja: conexiones sobre todo locales.
31	0.182	0.138	0.042	Grado bajo-moderado y betweenness relativamente alta: posible puente estructural.
3	0.182	0.012	0.037	Mismo grado que el 31, pero casi sin betweenness.

Lo más útil para tu informe: los nodos 31 y 3 tienen el mismo grado (0.182), pero betweenness muy distinta (0.138 frente a 0.012). Esto muestra que tener muchas conexiones no equivale a conectar regiones de la red. El 31 se ajusta a “grado moderado y betweenness alta” de tu tabla; el 1, a “grado moderado y betweenness baja”.

Qué mide cada métrica (úsalo al redactar):

Grado: proporción de los demás nodos con los que el nodo tiene un lazo directo.
Betweenness: proporción de caminos más cortos entre pares de nodos que pasan por él.
PageRank: importancia estructural según recibe lazos de nodos que también están bien conectados.

Párrafo de ejemplo, respetando la prohibición:

Según el grado, el nodo 33 es el que tiene más lazos directos (0.515). El nodo 0 tiene la mayor betweenness (0.438), lo que indica que muchos caminos más cortos de la red pasan por él, y es un posible puente estructural. Los nodos 31 y 3 comparten el mismo grado (0.182), pero el 31 tiene una betweenness mucho mayor (0.138 frente a 0.012), por lo que tener pocas conexiones no impide ocupar una posición de puente. Estas métricas describen posición en la red, no la influencia real de una persona.

Un límite: son 34 nodos de una sola red. Las comunidades (3, modularidad 0.411) son una agrupación del algoritmo para este grafo, no identidades sociales. Si quieres, te redacto la sección completa del informe, con la tabla y el análisis de comunidades.

In [ ]:
Resultado de embeddings:
Se entrenó Word2Vec (vector_size=60, min_count=1, seed=42, epochs=30) con demo_text.csv. Para la palabra "servicio", con window=5 los vecinos más cercanos fueron: en (0.307), las (0.298), frustrante (0.209), claro (0.190), el (0.170), confuso (0.161), terminó (0.144), registró (0.121), funcionó (0.109) y expediente (0.108). Al cambiar solo window a 2, los 10 vecinos y su orden fueron los mismos; variaron solo los decimales (por ejemplo, "en": 0.307 a 0.306).

Evidencia de cobertura:
Vocabulario de 54 palabras y cobertura de 1.0, igual con window=5 y window=2. La cobertura es 1.0 porque min_count=1 no descarta ninguna palabra, así que esa cifra no mide la calidad del corpus. El corpus es muy pequeño y sus frases son cortas.

Resultado estructural de la red:
La red analizada es el grafo del club de karate de Zachary (nx.karate_club_graph()), con 34 nodos y 78 aristas. El algoritmo greedy_modularity_communities detectó 3 comunidades con modularidad de 0.411.

Dos métricas comparadas:
Grado y betweenness. Los nodos 31 y 3 tienen el mismo grado (0.182), pero betweenness muy distinta: 0.138 el nodo 31 y 0.012 el nodo 3. El nodo 0 tiene la mayor betweenness (0.438) y el nodo 33 el mayor grado (0.515) y PageRank (0.097).

Interpretación permitida:
El grado mide la proporción de nodos con los que un nodo tiene un lazo directo. La betweenness mide la proporción de caminos más cortos que pasan por él. Que el nodo 31 tenga grado moderado y betweenness relativamente alta es consistente con que ocupe una posición de puente estructural entre grupos. Que el nodo 3 tenga el mismo grado y betweenness casi nula indica que tener conexiones no implica conectar regiones de la red. Las comunidades son agrupaciones de este algoritmo sobre esta red.

Interpretación que NO puede sostenerse:
No puede afirmarse que el nodo 33 o el 0 sea "la persona más influyente": las métricas describen posición estructural, no influencia real. Tampoco que las comunidades sean identidades sociales o temas. En embeddings, no puede concluirse que window=2 y window=5 capturen el mismo contexto en general: solo que, con este corpus de frases cortas y 54 palabras, no se observó diferencia. Los vecinos incluyen palabras vacías ("en", "las", "el") y similitudes bajas (máximo 0.31), por lo que son poco fiables.

Siguiente experimento:
Repetir con window=5 y min_count=2 para ver cómo cambian el vocabulario, la cobertura y los vecinos de "servicio". Además, entrenar Word2Vec con un corpus mayor y de frases más largas, y comparar window=2 frente a window=10, para comprobar si aparece la diferencia entre contexto sintáctico y temático.